# GNN–BERT Music Context: End-to-End Demo

This notebook demonstrates one end-to-end inference example using the trained **GNN–BERT cross-attention fusion model**.

**Pipeline:** GTZAN audio graph + paired MusicCaps caption → GraphSAGE + DistilBERT → cross-attention fusion → genre prediction.

This is an **inference-only demo**. It loads the trained checkpoint and does not retrain the model.


## 1. Project setup

Run this notebook from inside the repository after installing the dependencies in `requirements.txt`.

The notebook automatically searches the current directory and its parent directories to locate the repository root, so it works whether Jupyter is started from the repository root or from `notebooks/`.

The paths below match the project files used by the final fusion/case-study code:
- `data/processed/splits/fusion_dataset.csv`
- `data/models/best_fusion.pt`
- `src.models.fusion_model.FusionModel`
- `src.dataset.fusion_dataloader.load_graph`


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch
from transformers import DistilBertTokenizer

# Find the repository root regardless of whether Jupyter is started from
# the repository root, notebooks/, or another directory inside the repo.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (
        (candidate / "src" / "models" / "fusion_model.py").exists()
        and (candidate / "src" / "dataset" / "fusion_dataloader.py").exists()
        and (candidate / "data" / "processed" / "splits" / "fusion_dataset.csv").exists()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the GNN-BERT-Music-Context repository root. "
        "Run this notebook from inside the project repository."
    )

sys.path.insert(0, str(PROJECT_ROOT))

from src.models.fusion_model import FusionModel
from src.dataset.fusion_dataloader import load_graph

TEST_CSV = PROJECT_ROOT / "data/processed/splits/fusion_dataset.csv"
MODEL_PATH = PROJECT_ROOT / "data/models/best_fusion.pt"

MAX_LENGTH = 256
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("Dataset:", TEST_CSV)
print("Checkpoint:", MODEL_PATH)


## 2. Load the held-out multimodal test set

In [ ]:
if not TEST_CSV.exists():
    raise FileNotFoundError(f"Missing fusion dataset: {TEST_CSV}")

df = pd.read_csv(TEST_CSV)

test_df = df[df["split"] == "test"].reset_index(drop=True)

print(f"Fusion test samples: {len(test_df)}")
print("Columns:", list(test_df.columns))


## 3. Load the trained GNN–BERT fusion model

The final cross-attention model used in the project has:
- Graph embedding size: 64
- DistilBERT hidden size: 768
- Cross-attention projection size: 128
- Fusion hidden size: 256
- Output classes: 10 GTZAN genres

The checkpoint is loaded exactly as in the project's evaluation/case-study code.


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing trained checkpoint: {MODEL_PATH}")

model = FusionModel(
    graph_dim=64,
    text_dim=768,
    attention_dim=128,
    hidden_dim=256,
    num_classes=10,
    freeze_bert=False
).to(device)

state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("✓ Trained FusionModel loaded")


## 4. Select one test example

The example is taken from the held-out multimodal test split. The graph and caption are already paired according to the project's genre-based pairing protocol.

This pairing should **not** be interpreted as an exact recording-level GTZAN–MusicCaps correspondence.


In [ ]:
# Use the first held-out test example.
example = test_df.iloc[0]

true_genre = str(example["genre"])
caption = str(example["caption"])
graph_path = Path(str(example["graph_path"]))

# Make relative graph paths robust when the CSV stores paths relative to the repo.
if not graph_path.is_absolute():
    graph_path = PROJECT_ROOT / graph_path

print("True genre:", true_genre)
print("\nCaption:")
print(caption)
print("\nGraph path:")
print(graph_path)


## 5. Load the audio graph and tokenize the caption

In [ ]:
if not graph_path.exists():
    raise FileNotFoundError(f"Graph file not found: {graph_path}")

graph = load_graph(str(graph_path))

# One graph is treated as a batch of size 1.
graph.batch = torch.zeros(
    graph.x.size(0),
    dtype=torch.long
)

graph = graph.to(device)

encoded = tokenizer(
    caption,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

print("Graph node-feature shape:", tuple(graph.x.shape))
print("Token tensor shape:", tuple(input_ids.shape))


## 6. Run end-to-end inference

In [ ]:
with torch.no_grad():
    (
        logits,
        graph_embedding,
        text_hidden_states,
        attention_weights,
        z
    ) = model(
        graph,
        input_ids,
        attention_mask
    )

probabilities = torch.softmax(logits, dim=1)
prediction_index = int(logits.argmax(dim=1).item())
predicted_genre = GENRES[prediction_index]
confidence = float(probabilities[0, prediction_index].item())

print("True genre:     ", true_genre)
print("Predicted genre:", predicted_genre)
print("Confidence:     ", f"{confidence:.4f}")
print("Fused vector:   ", tuple(z.shape))


## 7. Show the prediction probabilities

The highest-probability genre is the model's final prediction for this graph–caption pair.


In [ ]:
prob_table = pd.DataFrame({
    "genre": GENRES,
    "probability": probabilities[0].detach().cpu().numpy()
}).sort_values("probability", ascending=False).reset_index(drop=True)

prob_table["probability"] = prob_table["probability"].round(4)
prob_table


## Demo result

This single example demonstrates the required end-to-end inference path:

**audio graph → GraphSAGE representation + caption → DistilBERT representation → cross-attention fusion → genre prediction.**

The notebook uses the already-trained model and a held-out test example; no training is performed during the demo.
